# ReSnap — Evaluasi Final Test Set, Ablasi Preprocessing, Export ONNX

Notebook ini menutup deliverable Modul 2 yang belum ada:
1. Evaluasi **test set** (654 foto) yang selama ini tidak disentuh. Jalankan **sekali** untuk laporan; jangan ubah model setelah melihat skornya.
2. **Ablasi**: skor model pada foto mentah vs foto yang melewati denoise + CLAHE dari `src/preprocessing.py`. Hasilnya menentukan default toggle di `app.py`.
3. Export `best.onnx` + grafik PR curve, F1 curve, confusion matrix untuk laporan.

Pilih `MODEL` di sel konfigurasi (v4 hanya bisa dipakai bila sudah diterima di `ReSnap_Retrain_V4.ipynb`). Sebelum mulai: upload folder repo `resnap/` ke `Tugas Akhir/resnap` di Drive. Runtime: GPU T4.

In [ ]:
%pip -q install ultralytics==8.3.228 "onnx>=1.12.0,<=1.19.1" "onnxslim>=0.1.71"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path
import yaml

PROJECT = Path('/content/drive/MyDrive/Brawijaya/MATKUL/SEM 5/COMVIS/Tugas Akhir')
REPO = PROJECT / 'resnap'
CANDIDATE = PROJECT / 'review_anotasi/tambahan_foto_udang_20261120_225843_394191/ReSnap_Kandidat_Tambah_Foto_Udang.zip'
CANDIDATE_SHA = '58d62af77338faa546fff2a77d0bc102b24956dc13fc8065a619dc7145b8e04c'
MODEL = 'v4'   # 'v4' bila notebook retrain menerima v4, selain itu 'v3'
RUNS = {'v3': 'resnap_yolov8n_30kelas_v3_udang_b1', 'v4': 'resnap_yolov8s_30kelas_v4_oi'}
WEIGHTS = PROJECT / 'training_yolov8' / RUNS[MODEL] / 'weights/best.pt'
OUT = PROJECT / f'evaluasi_final_{MODEL}'

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''):
            h.update(b)
    return h.hexdigest()

for p in (REPO / 'scripts/evaluate_export.py', CANDIDATE, WEIGHTS):
    assert p.exists(), f'Tidak ditemukan: {p}'
if MODEL == 'v3':
    assert sha(WEIGHTS) == 'ac24d434cc97d12f3fd3ed204d2fefda8cccfc322422a8c55672190a0615da98', 'best.pt bukan checkpoint v3'
else:
    keputusan = json.loads((PROJECT / 'evaluasi_v4/keputusan.json').read_text())
    assert keputusan['accept'], 'v4 belum diterima di notebook retrain; pakai MODEL = v3'
    done = json.loads((WEIGHTS.parent.parent / 'training_complete.json').read_text())
    assert sha(WEIGHTS) == done['weights_sha256'], 'best.pt v4 berubah sejak training selesai'
print('Model:', MODEL, WEIGHTS)
print('OK')

## 1. Ekstrak val + test dari ZIP kandidat v3

In [ ]:
work = Path('/content/resnap_final')
local_zip = work / 'candidate.zip'
work.mkdir(exist_ok=True)
if not local_zip.exists() or sha(local_zip) != CANDIDATE_SHA:
    shutil.copyfile(CANDIDATE, local_zip)
assert sha(local_zip) == CANDIDATE_SHA, 'Checksum ZIP tidak cocok'

data = work / 'data'
with zipfile.ZipFile(local_zip) as z:
    catalog = json.loads(z.read('reports/class_catalog.json'))
    names = {int(r['id']): r['name'] for r in catalog}
    members = [n for n in z.namelist() if n.startswith(('val/', 'test/')) and not n.endswith('/')]
    z.extractall(data, members)

counts = {s: len(list((data / s / 'images').iterdir())) for s in ('val', 'test')}
print(counts)
assert counts == {'val': 1069, 'test': 654}, counts
(data / 'data.yaml').write_text(yaml.safe_dump(dict(
    path=str(data), train='val/images', val='val/images', test='test/images', names=names),
    sort_keys=False), encoding='utf-8')

## 2. Evaluasi test + ablasi + ONNX

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(REPO / 'scripts/evaluate_export.py'), '--weights', str(WEIGHTS),
       '--data', str(data / 'data.yaml'), '--split', 'test', '--out', str(OUT / 'test'),
       '--device', '0', '--ablation', '--onnx']
subprocess.run(cmd, check=True, cwd=REPO)

Ablasi juga di val, supaya keputusan default preprocessing tidak hanya bersandar pada test.

In [ ]:
cmd = [sys.executable, str(REPO / 'scripts/evaluate_export.py'), '--weights', str(WEIGHTS),
       '--data', str(data / 'data.yaml'), '--split', 'val', '--out', str(OUT / 'val'),
       '--device', '0', '--ablation']
subprocess.run(cmd, check=True, cwd=REPO)

## 3. Ringkasan untuk laporan

In [ ]:
import pandas as pd
from IPython.display import Image, display

test = json.loads((OUT / 'test/ringkasan.json').read_text())
val = json.loads((OUT / 'val/ringkasan.json').read_text())
cols = ['images', 'precision', 'recall', 'f1', 'mAP50', 'mAP75', 'mAP50_95']
table = pd.DataFrame({
    'val (mentah)': val['raw'], 'val (denoise+CLAHE)': val['enhanced'],
    'test (mentah)': test['raw'], 'test (denoise+CLAHE)': test['enhanced'],
}).loc[cols]
display(table.style.format(precision=3))
print('Selisih test (poin persen):', test['delta_pp'])
print('Selisih val  (poin persen):', val['delta_pp'])

per_class = pd.read_csv(OUT / 'test/test_raw/per_class.csv').sort_values('AP50')
display(per_class.head(8).style.format(precision=3).set_caption('8 kelas terlemah di test'))
for f in ('confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png'):
    display(Image(str(OUT / 'test/test_raw' / f), width=700))

**Membaca hasil ablasi:** model dilatih dengan foto mentah, jadi skor dengan denoise + CLAHE biasanya turun. Kalau turun, pertahankan default `app.py` (toggle mati) dan tulis di laporan bahwa CLAHE dipakai sebagai opsi untuk foto redup, bukan default. Kalau naik di val **dan** test, nyalakan default toggle.